<img src="https://backend.burla.dev/static/welcome.svg" width="45%">

#### This notebook scans **36 months of NYC taxi parquet** (2022–2024) across 12 cloud workers and maps which neighborhoods are *emergent*, *ghost*, or *recovering*. In about 3 minutes. <br/> It's only 4 steps — follow along!

### What is Burla?

Burla is the simplest way to run Python across hundreds or thousands of cloud machines.
One function — `remote_parallel_map` — fans your code out across isolated Docker containers, grows the cluster on demand, and streams results back. Missing packages on the workers? Burla auto-installs them.

No Dockerfiles, no cluster YAML, no queue. Just Python.  
Burla is free to try — this whole notebook runs on machines we spin up for you.

### What will this demo do?

The full demo processes **2.76 billion NYC for-hire trips** — every yellow cab, green cab, and Uber/Lyft pickup the city has ever logged (2011-01 → 2024-12) — across 168 monthly parquet files, on up to 168 Burla workers, in ~15 seconds of wall-clock. It categorizes each of NYC's 264 taxi zones as `stable`, `emergent`, `cooling`, `warming`, `ghost`, or `resurrected`. See the [live site](https://burla-cloud.github.io/nyc-ghost-neighborhoods/).

In this notebook we run the same per-zone aggregation on **36 months** of yellow-cab data (2022-01 → 2024-12) across **12 Burla workers**. Each worker streams one month's parquet, aggregates trips by pickup zone, and returns a 263-row dict. The client stitches the monthly totals into a time series per zone and ranks emergent / fastest-growing zones.

## 1)&nbsp; Boot some machines (12+ CPUs):
&nbsp;&nbsp;&nbsp;&nbsp;Sign in using the button below, then hit the **`⏻ Start`** button on your dashboard homepage.  
&nbsp;&nbsp;&nbsp;&nbsp;This will boot a small cluster in Google Cloud for you. Burla is free to try so this is on us!

&nbsp;&nbsp;
<a href="https://login.burla.dev">
    <img src="https://i.ibb.co/QFNncHcp/Clean-Shot-2026-03-19-at-18-13-07.jpg" width="28%">
</a>

## 2)&nbsp; Install the Python package:

In [ ]:
!uv pip install burla

## 3)&nbsp; Authorize this computer:

In [ ]:
!burla login

## 4)&nbsp; Scan 36 months of NYC taxi parquet in parallel 🚀

We use the NYC TLC's public CloudFront distribution — no AWS credentials needed, just HTTPS. Each Burla worker downloads one month's yellow-cab parquet (50–100 MB) and returns per-zone trip counts.

In [ ]:
import pyarrow.parquet as pq  # noqa: F401  # top-level import -> Burla installs pyarrow on workers
from burla import remote_parallel_map

MONTHS = [(y, m) for y in (2022, 2023, 2024) for m in range(1, 13)]
urls = [(y, m, f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{y}-{m:02d}.parquet")
        for (y, m) in MONTHS]
print(f"scanning {len(urls)} months on {len(urls)} workers")


def per_zone_count(job: tuple) -> dict:
    import io
    import urllib.request
    import pyarrow.parquet as pq
    from collections import Counter

    year, month, url = job
    with urllib.request.urlopen(url, timeout=120) as r:
        body = r.read()

    # Read just the pickup zone column
    table = pq.read_table(io.BytesIO(body), columns=["PULocationID"])
    zones = table.column("PULocationID").to_pylist()
    counts = Counter(z for z in zones if z is not None)
    return {
        "year": year,
        "month": month,
        "total_trips": len(zones),
        "zone_counts": dict(counts),
    }


# 36 months -> 36 tasks running on up to 36 workers in parallel
month_results = remote_parallel_map(per_zone_count, urls, func_cpu=1, func_ram=4, grow=True)
total_trips = sum(r["total_trips"] for r in month_results)
print(f"scanned {total_trips:,} taxi trips across {len(month_results)} monthly files")

### What just happened?

You just streamed ~3 GB of Parquet data (36 months × ~100 MB each), aggregated by pickup zone on each worker, and returned only a tiny 263-row dict per month. Total network egress from the workers back to the notebook was a few KB — not gigabytes.

The full 168-month / 2.76B-trip version is the same code plus the green-cab and FHV (Uber/Lyft) parquet URLs. Same shape, 4,000× more data, still ~15 seconds wall-clock.

### Stitch into a per-zone time series + find emergent zones

An emergent zone is one whose earliest-12-month average is tiny but whose latest-12-month average is significant. The full demo ranks zones by the ratio `recent / birth` — let's do the same here.

In [ ]:
import pandas as pd

# Build long-form DataFrame: (year, month, zone, trips)
rows = []
for r in month_results:
    if "error" in r:
        continue
    for zone, trips in r["zone_counts"].items():
        rows.append({"year": r["year"], "month": r["month"], "zone": zone, "trips": trips})
df = pd.DataFrame(rows)
df["ym"] = df["year"] * 100 + df["month"]

# For each zone, compute first-12-months mean vs last-12-months mean
pivot = df.pivot_table(index="zone", columns="ym", values="trips", fill_value=0)
pivot = pivot.sort_index(axis=1)
first12 = pivot.iloc[:, :12].mean(axis=1)
last12 = pivot.iloc[:, -12:].mean(axis=1)
growth = (last12 / first12.replace(0, 0.5)).rename("growth")

summary = pd.concat([first12.rename("first12_mean"), last12.rename("last12_mean"), growth], axis=1)
summary["total_trips"] = pivot.sum(axis=1)

print("=" * 60)
print("TOP-10 EMERGENT ZONES (growth = last-12mo / first-12mo)")
print("=" * 60)
emergent = summary[summary["total_trips"] > 1000].sort_values("growth", ascending=False).head(10)
print(emergent.round(1).to_string())

print()
print("=" * 60)
print("TOP-10 DECLINING ZONES (lowest last-12mo / first-12mo ratio)")
print("=" * 60)
declining = summary[summary["total_trips"] > 10000].sort_values("growth", ascending=True).head(10)
print(declining.round(1).to_string())

### Try this next

- Extend `MONTHS` back to 2011-01-01 (168 months, same as the full demo).
- Add green cab (`green_tripdata_*`) and FHV (`fhvhv_tripdata_*`) URLs — that's the full 371-file run.
- Join `PULocationID` with NYC TLC's taxi zone lookup CSV to put names on the rankings.
- Swap the metric for `revenue_sum` to find zones where fare economics have shifted.

## Thank you for trying Burla! ❤️

### Run the full 2.76-billion-trip version

See [`nyc_ghost_neighborhoods.py`](https://github.com/Burla-Cloud/nyc-ghost-neighborhoods/blob/main/nyc_ghost_neighborhoods.py) in this repo — 168 months × yellow + green + FHV + HVFHS Parquet, up to 168 Burla workers, 15 seconds of wall-clock.

### Run this on your laptop 🧑‍💻

1. `pip install burla`
2. `burla login`
3. Run the same example code from above ☝️

Congrats! Your laptop now effectively has 1000+ CPUs, and 4000G of RAM &nbsp; : )

<br/>

### Any way we can help? Lets chat!

Schedule a meeting: https://cal.com/jakez  
Send me an email: jake@burla.dev